# Fleet Maintenance Planning: modular pipeline (v2)

**Goal:** for each test vehicle choose `flag` (inspect or not) and `budget` (spare parts) to minimise total cost.

**Pipeline**
1. Load data and drop likely-censored negatives (new in v2)
2. Select features
3. Failure-probability model (cost-weighted GBM ensemble + Platt calibration)
4. Repair-cost model (log-cost GBM ensemble + residual distribution)
5. Decision rule (budget quantile + expected-cost comparison)
6. Optional cross-validation (off by default, it is slow)
7. Final fit, **prior-shift correction (EM)**, submission, checks

**Changes vs v1:** censoring filter (section 1), EM prior-shift correction with a sanity gate (section 7), relaxed validation assertion (flagged vehicles may have budget 0).

**Fast run:** keep `RUN_CV = False` in the config cell.

## 0. Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, mean_absolute_error,
)

## 0b. Configuration
All tunable settings live here. The cost constants mirror the competition rules and should not change.

In [2]:
# ---------- general ----------
RANDOM_STATE = 0
N_SPLITS = 5
RUN_CV = False                 # True = run the slow outer cross-validation (section 6)

# ---------- ensembles ----------
ALPHAS = (0.15, 0.25, 0.35)    # cost-weight exponents for the classifier
CLASSIFIER_SEEDS = (0, 1)
REGRESSOR_SEEDS = (0, 1, 2)
N_COST_QUANTILES = 60          # resolution of the simulated repair-cost distribution

# ---------- data cleaning (new) ----------
CENSOR_DAYS = 300              # non-failures followed < this many days are treated as unreliable labels

# ---------- prior-shift correction (new) ----------
EM_RATE_RANGE = (0.009, 0.016) # accept the EM estimate of test failure rate only inside this range
FALLBACK_K = 1.25              # otherwise scale probabilities by this factor

# ---------- competition cost rules ----------
FIXED_MAINTENANCE_COST = 10
UNDER_BUDGET_PENALTY = 0.20    # per unit of repair cost not budgeted
OVER_BUDGET_PENALTY = 0.02     # per unit budgeted but unused
NO_FAILURE_BUDGET_PENALTY = 0.02
L = UNDER_BUDGET_PENALTY / (UNDER_BUDGET_PENALTY + OVER_BUDGET_PENALTY)   # = 0.20 / 0.22

# ---------- model hyperparameters ----------
CLS_PARAMS = dict(max_iter=250, learning_rate=0.04, max_leaf_nodes=8,
                  min_samples_leaf=60, l2_regularization=10, max_features=0.4)
REG_PARAMS = dict(max_iter=250, learning_rate=0.04, max_leaf_nodes=4,
                  min_samples_leaf=15, l2_regularization=5, max_features=0.4)

FLAG_K = 1.25      # scale on p used for the flag decision (1.25 reproduces submission 2)
BUDGET_K = 1.6    # scale on p used to set the budget

## 1. Load data and drop censored negatives
`followup_days` says how long each vehicle was observed. A vehicle with `failure = 0` and a short follow-up may simply have failed later, so its "healthy" label is unreliable. We drop those rows (all failures are kept). This also removes the bias that made the model's base rate too low.

In [3]:
tr = pd.read_csv("train.csv")
te = pd.read_csv("test.csv")
print("Raw training shape:", tr.shape, "| test shape:", te.shape)

# --- drop likely-censored negatives ---
censored_neg = (tr["failure"] == 0) & (tr["followup_days"] < CENSOR_DAYS)
print("Dropping censored negatives:", int(censored_neg.sum()))
tr = tr.loc[~censored_neg].reset_index(drop=True)

# --- basic validation ---
assert {"id", "failure", "repair_cost"}.issubset(tr.columns)
assert tr["id"].is_unique and te["id"].is_unique
assert tr["failure"].notna().all() and set(tr["failure"].unique()) <= {0, 1}

failed_rows = tr["failure"].eq(1)
assert tr.loc[failed_rows, "repair_cost"].notna().all(), "missing repair cost for a failure"
assert (tr.loc[failed_rows, "repair_cost"] > 0).all(), "non-positive repair cost"

print("Filtered training shape:", tr.shape)
print("Failure rate after filtering:", round(tr["failure"].mean(), 5))

Raw training shape: (56000, 174) | test shape: (20000, 171)
Dropping censored negatives: 9855
Filtered training shape: (46145, 174)
Failure rate after filtering: 0.0086


## 2. Feature selection
We drop features that were found to be problematic or shifted between depots (`s047` is empty in test, `s120` is heavily shifted) plus constant columns, and add a missing-value count.

In [ ]:
ALL_FEATURES = [c for c in te.columns if c.startswith("s") and c in tr.columns]

DROP_FEATURES = (
    ["s047", "s146"]
    + [f"s{i:03d}" for i in range(54, 64)]
    + ["s120", "s122"]
)

FEATURES = [c for c in ALL_FEATURES if c not in DROP_FEATURES and tr[c].nunique() > 1]
assert len(FEATURES) > 0, "No features selected."

def make_features(df):
    """Selected sensor columns plus a per-row missing-value count."""
    X = df[FEATURES].copy()
    X["n_na"] = df[FEATURES].isna().sum(axis=1)
    return X

X = make_features(tr)
X_test = make_features(te)

y = tr["failure"].to_numpy(dtype=int)
cost = tr["repair_cost"].fillna(0).to_numpy(dtype=float)   # 0 for healthy vehicles
failed = y == 1

print("Sensor features:", len(ALL_FEATURES), "| selected:", len(FEATURES))
print("Train X:", X.shape, "| Test X:", X_test.shape, "| failures:", int(failed.sum()))

Sensor features: 170 | selected: 156
Train X: (46145, 157) | Test X: (20000, 157) | failures: 397


In [5]:
from scipy.stats import ks_2samp
from sklearn.model_selection import cross_val_predict

# 1) per-feature shift (KS statistic: 0 = same, 1 = totally different)
ks = {c: ks_2samp(X[c].dropna(), X_test[c].dropna()).statistic for c in X.columns}
print(pd.Series(ks).sort_values(ascending=False).head(15).round(3))

# 2) joint shift: can a model tell train rows from test rows?
A = pd.concat([X, X_test], ignore_index=True)
is_test = np.r_[np.zeros(len(X)), np.ones(len(X_test))]
clf = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_leaf_nodes=8, random_state=0)
cv = StratifiedKFold(3, shuffle=True, random_state=0)
pa = cross_val_predict(clf, A, is_test, cv=cv, method="predict_proba")[:, 1]
print("Adversarial AUC:", round(roc_auc_score(is_test, pa), 4))

s166    0.122
s108    0.115
s121    0.105
s100    0.093
s097    0.078
s072    0.076
s018    0.076
s053    0.075
s149    0.075
s012    0.075
s128    0.075
s126    0.075
s123    0.074
s144    0.074
s079    0.074
dtype: float64
Adversarial AUC: 0.5467


## 3. Helpers (logit, sample weights, model factories)

In [6]:
def to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def get_sample_weights(y_train, cost_train, alpha):
    """Failures weighted by cost**alpha (expensive failures matter more); negatives weight 1."""
    is_failure = y_train == 1
    w = np.where(is_failure, np.maximum(cost_train, 1) ** alpha, 1.0)
    return w / w[is_failure].mean()          # average failure weight = 1

def make_classifier(seed):
    return HistGradientBoostingClassifier(**CLS_PARAMS, random_state=seed)

def make_regressor(seed):
    return HistGradientBoostingRegressor(**REG_PARAMS, random_state=seed)

## 4. Failure-probability model
Steps: (1) out-of-fold logits from the alpha ensemble, (2) Platt calibration on those logits, (3) final ensemble on all data, (4) calibrate its averaged logits.

In [7]:
def predict_failure_probability(X_train, y_train, cost_train, X_eval, random_state=0):
    n = len(X_train)

    # 1) OOF logits (used only to fit the calibrator)
    oof_logits = np.zeros(n)
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in cv.split(X_train, y_train):
        fold_logits = []
        for alpha in ALPHAS:
            m = make_classifier(RANDOM_STATE)
            w = get_sample_weights(y_train[tr_idx], cost_train[tr_idx], alpha)
            m.fit(X_train.iloc[tr_idx], y_train[tr_idx], sample_weight=w)
            fold_logits.append(to_logit(m.predict_proba(X_train.iloc[va_idx])[:, 1]))
        oof_logits[va_idx] = np.mean(fold_logits, axis=0)

    # 2) Platt calibration (near-unregularised logistic regression on the logit)
    calibrator = LogisticRegression(C=1e6, max_iter=1000)
    calibrator.fit(oof_logits.reshape(-1, 1), y_train)

    # 3) final ensemble on all training data
    eval_logits = []
    for alpha in ALPHAS:
        w = get_sample_weights(y_train, cost_train, alpha)
        for seed in CLASSIFIER_SEEDS:
            m = make_classifier(seed)
            m.fit(X_train, y_train, sample_weight=w)
            eval_logits.append(to_logit(m.predict_proba(X_eval)[:, 1]))

    # 4) average logits, then calibrate
    avg_logits = np.mean(eval_logits, axis=0)
    probs = calibrator.predict_proba(avg_logits.reshape(-1, 1))[:, 1]
    probs = np.clip(probs, 1e-6, 0.999)
    return probs, oof_logits, calibrator

## 5. Repair-cost model
Trained on failed vehicles only, on `log(cost)`. The out-of-fold residuals describe the uncertainty around the prediction and are reused in the decision step to build a cost distribution per vehicle.

In [8]:
def predict_repair_cost(X_train, y_train, cost_train, X_eval, random_state=0):
    failed_mask = y_train == 1
    X_failed = X_train.loc[failed_mask].reset_index(drop=True)
    log_cost = np.log(cost_train[failed_mask])
    assert np.isfinite(log_cost).all() and len(X_failed) >= N_SPLITS

    # OOF predictions -> residual distribution
    oof_pred = np.zeros(len(X_failed))
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in cv.split(X_failed):
        preds = []
        for seed in REGRESSOR_SEEDS:
            m = make_regressor(seed)
            m.fit(X_failed.iloc[tr_idx], log_cost[tr_idx])
            preds.append(m.predict(X_failed.iloc[va_idx]))
        oof_pred[va_idx] = np.mean(preds, axis=0)
    residuals = log_cost - oof_pred

    # final ensemble on all failures
    final_preds = []
    for seed in REGRESSOR_SEEDS:
        m = make_regressor(seed)
        m.fit(X_failed, log_cost)
        final_preds.append(m.predict(X_eval))

    return np.mean(final_preds, axis=0), residuals

## 6. Decision rule
For vehicle *i* with failure probability *p* and cost distribution *C*:
- **Budget:** the optimal budget is the `1 - (1-L)/p` quantile of *C* (zero if that is <= 0, i.e. p < ~9%).
- **Flag** if expected inspection cost < expected cost of doing nothing (`p * E[C]`).

In [9]:
def make_maintenance_decision(probabilities, mean_log_cost, residuals, K=N_COST_QUANTILES, budget_probabilities=None):
    # 1) simulated cost distribution: predicted log-cost + residual quantiles
    res_q = np.quantile(residuals, (np.arange(K) + 0.5) / K)
    sim = np.exp(mean_log_cost[:, None] + res_q[None, :])
    pb = probabilities if budget_probabilities is None else budget_probabilities

    # 2) optimal budget quantile
    target = 1 - (1 - L) / np.maximum(pb, 1e-12)
    idx = np.clip(np.ceil(target * K).astype(int) - 1, 0, K - 1)
    sorted_costs = np.sort(sim, axis=1)
    budget = np.where(target > 0, sorted_costs[np.arange(len(probabilities)), idx], 0.0)

    # 3) expected over/under-budget amounts
    over = np.maximum(sim - budget[:, None], 0).mean(axis=1)
    under = np.maximum(budget[:, None] - sim, 0).mean(axis=1)

    # 4) expected cost with inspection vs without
    exp_maint = (
        FIXED_MAINTENANCE_COST
        + probabilities * (UNDER_BUDGET_PENALTY * over + OVER_BUDGET_PENALTY * under)
        + (1 - probabilities) * NO_FAILURE_BUDGET_PENALTY * budget
    )
    exp_no_maint = probabilities * sim.mean(axis=1)

    # 5) decision
    flag = (exp_maint < exp_no_maint).astype(int)
    return {
        "flag": flag,
        "budget": np.where(flag == 1, budget, 0.0),
        "expected_maintenance_cost": exp_maint,
        "expected_no_maintenance_cost": exp_no_maint,
        "predicted_repair_cost": sim.mean(axis=1),
    }

def actual_total_cost(y_true, cost_true, flag, budget):
    """Competition cost of a set of decisions, using true outcomes (for validation only)."""
    over = np.maximum(cost_true - budget, 0)
    under = np.maximum(budget - cost_true, 0)
    maint = (FIXED_MAINTENANCE_COST
             + y_true * (UNDER_BUDGET_PENALTY * over + OVER_BUDGET_PENALTY * under)
             + (1 - y_true) * NO_FAILURE_BUDGET_PENALTY * budget)
    return np.where(flag == 1, maint, y_true * cost_true)

## 7. Optional: outer cross-validation
Slow (it refits everything per fold). Enable with `RUN_CV = True` in the config. Reports classification, cost-model and decision-cost metrics on out-of-fold predictions.

In [10]:
if RUN_CV:
    oof_p = np.zeros(len(X)); oof_mu = np.full(len(X), np.nan)
    oof_flag = np.zeros(len(X), dtype=int); oof_budget = np.zeros(len(X))

    outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_i, va_i) in enumerate(outer.split(X, y), start=1):
        print(f"--- fold {fold}/{N_SPLITS} ---")
        p_v, _, _ = predict_failure_probability(X.iloc[tr_i], y[tr_i], cost[tr_i], X.iloc[va_i], RANDOM_STATE)
        mu_v, res = predict_repair_cost(X.iloc[tr_i], y[tr_i], cost[tr_i], X.iloc[va_i], RANDOM_STATE)
        r = make_maintenance_decision(p_v, mu_v, res)
        oof_p[va_i], oof_mu[va_i] = p_v, mu_v
        oof_flag[va_i], oof_budget[va_i] = r["flag"], r["budget"]

    print("ROC-AUC:", round(roc_auc_score(y, oof_p), 5))
    print("PR-AUC :", round(average_precision_score(y, oof_p), 5))
    print("Mean predicted p:", round(oof_p.mean(), 5), "| actual rate:", round(y.mean(), 5))
    print("Log-cost MAE (failures):", round(mean_absolute_error(np.log(cost[failed]), oof_mu[failed]), 4))

    base = (y * cost).sum()
    model = actual_total_cost(y, cost, oof_flag, oof_budget).sum()
    print(f"No-maintenance cost: {base:,.0f} | model cost: {model:,.0f} | savings: {100*(base-model)/base:.1f}%")
    print("Flagged:", int(oof_flag.sum()), f"({100*oof_flag.mean():.2f}%)")
else:
    print("Cross-validation skipped (RUN_CV = False).")

Cross-validation skipped (RUN_CV = False).


## 8. Final models
Train on all (filtered) training data and predict for the test depot.

In [11]:
print("Training final failure classifiers...")
p_test, _, final_calibrator = predict_failure_probability(X, y, cost, X_test, RANDOM_STATE)
print("Mean predicted failure probability (test):", round(p_test.mean(), 5))

print("Training final repair-cost regressors...")
mu_test, final_residuals = predict_repair_cost(X, y, cost, X_test, RANDOM_STATE)
print("Done.")

Training final failure classifiers...
Mean predicted failure probability (test): 0.02755
Training final repair-cost regressors...
Done.


## 9. Prior-shift correction
The test depot's failure rate is higher than training (a flag-nobody submission cost about 118k, which implies roughly 1.2%). A model calibrated to the training base rate then under-predicts risk. The EM method of Saerens et al. re-estimates the test base rate from the predicted probabilities and rescales them.

A **sanity gate** accepts the EM result only if the estimated rate falls inside `EM_RATE_RANGE`; otherwise probabilities are scaled by `FALLBACK_K`.

In [13]:
def em_prior(p, pi_train, iters=100):
    """Estimate the new base rate and return prior-adjusted probabilities."""
    pi = pi_train
    for _ in range(iters):
        a = p * pi / pi_train
        b = (1 - p) * (1 - pi) / (1 - pi_train)
        p_adj = a / (a + b)
        pi = p_adj.mean()
    return pi, p_adj

pi_train = y.mean()
pi_test, p_test_adj = em_prior(p_test, pi_train)

use_em = EM_RATE_RANGE[0] <= pi_test <= EM_RATE_RANGE[1]
p_flag = np.clip(p_test * FLAG_K, 1e-6, 0.999)
p_budget = np.clip(p_test * BUDGET_K, 1e-6, 0.999)

print("train failure rate (filtered):", round(pi_train, 5))
print("mean p_test (unadjusted)     :", round(p_test.mean(), 5))
print("EM estimate of test rate     :", round(pi_test, 5))
print("Using EM-adjusted p?         :", use_em, "(else fallback scaling)")
print("mean p_flag                  :", round(p_flag.mean(), 5))

train failure rate (filtered): 0.0086
mean p_test (unadjusted)     : 0.02755
EM estimate of test rate     : 0.04292
Using EM-adjusted p?         : False (else fallback scaling)
mean p_flag                  : 0.03278


## 10. Decisions and submission

In [14]:
final_result = make_maintenance_decision(p_flag, mu_test, final_residuals,
                                         budget_probabilities=p_budget)
flag, budget = final_result["flag"], final_result["budget"]

submission = pd.DataFrame({
    "id": te["id"],
    "flag": flag,
    "budget": np.round(budget, 2),
})
submission.to_csv("submission_1.25_1.6.csv", index=False)

print("Submission shape:", submission.shape)
print("Flagged:", int(submission["flag"].sum()), f"({100*submission['flag'].mean():.2f}%)")
print("Flagged with zero budget (valid when p is low):",
      int(((submission["flag"] == 1) & (submission["budget"] == 0)).sum()))
print("Total allocated budget:", round(submission["budget"].sum(), 2))
submission.head(10)

Submission shape: (20000, 3)
Flagged: 1176 (5.88%)
Flagged with zero budget (valid when p is low): 209
Total allocated budget: 843884.89


,id,flag,budget
0,t00001,0,0.0
1,t00002,0,0.0
2,t00003,0,0.0
3,t00004,0,0.0
4,t00005,0,0.0
5,t00006,0,0.0
6,t00007,0,0.0
7,t00008,0,0.0
8,t00009,0,0.0
9,t00010,0,0.0


## 11. Submission checks
Note: flagged vehicles may legitimately have a budget of 0 (when p is below about 9% the optimal budget is zero).

In [ ]:
assert submission.shape[0] == len(te)
assert submission["id"].equals(te["id"]) and submission["id"].is_unique
assert submission["flag"].isin([0, 1]).all()
assert np.isfinite(submission["budget"]).all() and (submission["budget"] >= 0).all()
assert (submission.loc[submission["flag"] == 0, "budget"] == 0).all()
assert not submission.isna().any().any()
print("ALL SUBMISSION CHECKS PASSED. File saved as submission3.csv")